In [1]:
# 02_eda_sample_characteristics.ipynb
# Exploratory data analysis and sample characterisation for the analytic dataset.
# Produces (i) a baseline characteristics table by sex, (ii) predictor-outcome
# association tests (t-test / chi-square) by sex, and (iii) a grayscale figure of
# diabetes prevalence by age band and sex. All figures are grayscale seaborn,
# dpi 600, saved as both PNG and PDF, with no embedded titles/captions.

import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

# ---- Paths ----
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "data"
TAB = ROOT / "results" / "tables"
FIG = ROOT / "results" / "figures"

# ---- Grayscale plotting style (shared convention across all notebooks) ----
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"savefig.dpi": 600, "figure.dpi": 120, "pdf.fonttype": 42,
                     "ps.fonttype": 42, "font.size": 11, "axes.edgecolor": "0.2",
                     "axes.linewidth": 0.8, "grid.color": "0.85"})
GREY_DARK, GREY_LIGHT = "#3b3b3b", "#a9a9a9"
SEX_GREY = {"Male": GREY_DARK, "Female": GREY_LIGHT}

def savefig(fig, name):
    fig.savefig(FIG / f"{name}.png", dpi=600, bbox_inches="tight")
    fig.savefig(FIG / f"{name}.pdf", bbox_inches="tight")
    plt.close(fig)

df = pd.read_parquet(DATA / "analytic_dataset.parquet")
CONT = ["age", "HE_BMI", "HE_wc", "HE_wt", "HE_ht", "N_EN", "N_CHO", "N_SUGAR", "N_CHOL"]
CAT = ["sm_presnt", "dr_month", "pa_aerobic", "L_BR_FQ", "edu", "ho_incm", "D_1_1"]

# ---- (i) Baseline characteristics by sex ----
rows = []
for v in CONT:
    m, f = df[df.sex == "Male"][v], df[df.sex == "Female"][v]
    rows.append([v, "mean(SD)", f"{m.mean():.1f} ({m.std():.1f})",
                 f"{f.mean():.1f} ({f.std():.1f})"])
for v in CAT:
    m, f = df[df.sex == "Male"][v], df[df.sex == "Female"][v]
    rows.append([v, "mean(SD)", f"{m.mean():.2f} ({m.std():.2f})",
                 f"{f.mean():.2f} ({f.std():.2f})"])
base = pd.DataFrame(rows, columns=["variable", "summary", "Male", "Female"])
base.to_csv(TAB / "table2_baseline_characteristics.csv", index=False)
print("Baseline characteristics by sex:")
print(base.to_string(index=False))

# ---- (ii) Predictor-outcome association by sex ----
assoc = []
for sx in ["Male", "Female"]:
    s = df[df.sex == sx]
    for v in CONT:
        a = s[s.diabetes == 1][v]; b = s[s.diabetes == 0][v]
        t, p = stats.ttest_ind(a, b, equal_var=False)
        assoc.append([sx, v, "t-test", round(t, 2), p])
    for v in CAT:
        ct = pd.crosstab(s[v], s.diabetes)
        chi2, p, dof, _ = stats.chi2_contingency(ct)
        assoc.append([sx, v, "chi2", round(chi2, 2), p])
ad = pd.DataFrame(assoc, columns=["sex", "variable", "test", "statistic", "p_value"])
ad["p_value"] = ad["p_value"].apply(lambda x: f"{x:.2e}")
ad.to_csv(TAB / "table3_predictor_outcome_association.csv", index=False)
print("\nStrongest categorical associations (by |chi2|):")
print(ad[ad.test == "chi2"].to_string(index=False))

# ---- (iii) Diabetes prevalence by age band and sex ----
bins = [19, 30, 40, 50, 60, 70, 120]
lab = ["19-29", "30-39", "40-49", "50-59", "60-69", "70+"]
df["ageband"] = pd.cut(df.age, bins=bins, labels=lab, right=False)
prev = (df.groupby(["ageband", "sex"]).diabetes.mean().mul(100).reset_index())

fig, ax = plt.subplots(figsize=(7, 4.2))
sns.barplot(data=prev, x="ageband", y="diabetes", hue="sex",
            palette=SEX_GREY, edgecolor="black", linewidth=0.7, ax=ax)
# distinguish bars by hatch in addition to grayscale
for bars, h in zip([ax.containers[0], ax.containers[1]], ["", "///"]):
    for b in bars:
        b.set_hatch(h)
ax.set_xlabel("Age band (years)"); ax.set_ylabel("Diabetes prevalence (%)")
ax.legend(title="", frameon=False)
savefig(fig, "fig_prevalence_by_age_sex")

prev.to_csv(TAB / "table_prevalence_by_age_sex.csv", index=False)
print("\nPrevalence by age band and sex (%):")
print(prev.pivot(index="ageband", columns="sex", values="diabetes").round(1).to_string())
print("\nSaved EDA tables and fig_prevalence_by_age_sex.(png|pdf)")


Baseline characteristics by sex:
  variable  summary           Male         Female
       age mean(SD)    52.8 (17.2)    53.1 (16.6)
    HE_BMI mean(SD)     24.8 (3.6)     23.6 (3.8)
     HE_wc mean(SD)     88.5 (9.6)    80.4 (10.4)
     HE_wt mean(SD)    72.6 (12.7)    58.8 (10.1)
     HE_ht mean(SD)    170.9 (6.8)    157.9 (6.4)
      N_EN mean(SD) 2131.4 (867.0) 1549.1 (615.9)
     N_CHO mean(SD)  297.2 (113.3)   229.3 (93.0)
   N_SUGAR mean(SD)    60.2 (43.2)    54.3 (37.4)
    N_CHOL mean(SD)  289.4 (231.8)  221.6 (180.9)
 sm_presnt mean(SD)    0.29 (0.45)    0.05 (0.21)
  dr_month mean(SD)    0.65 (0.48)    0.39 (0.49)
pa_aerobic mean(SD)    0.47 (0.50)    0.42 (0.49)
   L_BR_FQ mean(SD)    1.88 (1.21)    1.86 (1.19)
       edu mean(SD)    3.09 (1.00)    2.86 (1.13)
   ho_incm mean(SD)    2.75 (1.08)    2.64 (1.09)
     D_1_1 mean(SD)    2.76 (0.86)    2.92 (0.87)

Strongest categorical associations (by |chi2|):
   sex   variable test  statistic   p_value
  Male  sm_presnt chi2  


Prevalence by age band and sex (%):
sex      Female  Male
ageband              
19-29       1.0   1.6
30-39       2.1   4.6
40-49       4.4  11.8
50-59      12.7  22.1
60-69      21.3  30.6
70+        28.6  30.5

Saved EDA tables and fig_prevalence_by_age_sex.(png|pdf)
